# 04 · Marginal split-conformal v1

Freezes marginal split-conformal thresholds on the reserved conformal-calibration role.

**Checkpoint date:** 2026-10-05.

In [ ]:
#!/usr/bin/env python3
"""
Freeze 72-hour split-conformal classification thresholds for SHARP, AIA,
and fixed 0.5/0.5 fusion using ONLY the reserved conformal_calibration role.

Safety / protocol:
- no model fitting or retraining
- no GPU use
- no deletion
- no GCS mutation
- does not read policy_validation, retrospective_cycle25, or supplementary_2026 outcomes
- probability calibration is identity/raw for all three branches, as previously frozen
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import numpy as np
import pandas as pd

MASTER = Path("/home/abmoses2000/graybox_aia72_master_predictions_v2_20261005/graybox_aia72_master_predictions.csv.gz")
MASTER_SHA = "17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46"

SHARP = Path("/home/abmoses2000/sharp72_predictions_frozen_20261002.csv.gz")
SHARP_SHA = "7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b"

CAL = Path("/home/abmoses2000/multimodal72_calibration_v1_20261005T082136654618Z")
SELECTION = CAL / "selection.json"

OUT = Path("/home/abmoses2000/graybox_aia72_conformal_v1_20261005")
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

ALPHAS = [0.10, 0.05, 0.20]
PRIMARY_ALPHA = 0.10
EXPECTED_ROWS = 4168
EXPECTED_POSITIVES = 384

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def require(cond, msg):
    if not cond:
        raise ValueError(msg)

require(MASTER.is_file(), f"Missing master: {MASTER}")
require(SHARP.is_file(), f"Missing SHARP predictions: {SHARP}")
require(SELECTION.is_file(), f"Missing saved probability calibration selection: {SELECTION}")

require(sha256(MASTER) == MASTER_SHA, "Frozen master SHA changed")
require(sha256(SHARP) == SHARP_SHA, "Frozen SHARP prediction SHA changed")

selection = json.loads(SELECTION.read_text())
require(
    selection["selected_methods"] == {"sharp": "raw", "aia": "raw", "sharp_aia": "raw"},
    "Probability-calibration choice changed"
)

master = pd.read_csv(MASTER, low_memory=False)
sharp = pd.read_csv(SHARP, low_memory=False)

# Read ONLY the reserved conformal-calibration role from both sources.
aia_c = master.loc[master["role"].astype(str) == "conformal_calibration"].copy()
sharp_c = sharp.loc[sharp["role"].astype(str) == "conformal_calibration"].copy()

require(len(aia_c) == EXPECTED_ROWS, f"AIA conformal rows changed: {len(aia_c)}")
require(len(sharp_c) == EXPECTED_ROWS, f"SHARP conformal rows changed: {len(sharp_c)}")
require(aia_c["forecast_case_id"].is_unique, "AIA conformal IDs are not unique")
require(sharp_c["forecast_case_id"].is_unique, "SHARP conformal IDs are not unique")

aia_c["forecast_case_id"] = aia_c["forecast_case_id"].astype(str)
sharp_c["forecast_case_id"] = sharp_c["forecast_case_id"].astype(str)

# Exact ordered identity is stronger than set equality.
require(
    aia_c["forecast_case_id"].tolist() == sharp_c["forecast_case_id"].tolist(),
    "SHARP and AIA conformal IDs/order differ"
)

# Authoritative labels must match.
aia_label = pd.to_numeric(aia_c["label"], errors="raise").astype(int).to_numpy()
sharp_label = pd.to_numeric(sharp_c["label"], errors="raise").astype(int).to_numpy()
require(np.array_equal(aia_label, sharp_label), "SHARP and AIA conformal labels disagree")
require(set(np.unique(aia_label)).issubset({0, 1}), "Labels are not binary")
require(int(aia_label.sum()) == EXPECTED_POSITIVES,
        f"Positive count changed: {int(aia_label.sum())}")

# Frozen raw probabilities.
require("probability" in aia_c.columns, "AIA master lacks ensemble probability column")
require("probability_gru_mean" in sharp_c.columns, "SHARP file lacks probability_gru_mean")

aia_p = pd.to_numeric(aia_c["probability"], errors="raise").to_numpy(float)
sharp_p = pd.to_numeric(sharp_c["probability_gru_mean"], errors="raise").to_numpy(float)

for name, p in [("aia", aia_p), ("sharp", sharp_p)]:
    require(np.isfinite(p).all(), f"{name} probabilities contain non-finite values")
    require(((0 <= p) & (p <= 1)).all(), f"{name} probabilities outside [0,1]")

fusion_p = 0.5 * sharp_p + 0.5 * aia_p

branches = {
    "sharp": sharp_p,
    "aia": aia_p,
    "sharp_aia": fusion_p,
}

def true_class_probability(p, y):
    return np.where(y == 1, p, 1.0 - p)

def finite_sample_qhat(scores, alpha):
    """
    Split-conformal finite-sample quantile:
      k = ceil((n + 1) * (1 - alpha))
    with k clipped to [1, n], then the k-th order statistic.
    """
    scores = np.asarray(scores, dtype=float)
    n = len(scores)
    require(n > 0, "Empty conformal score vector")
    k = int(math.ceil((n + 1) * (1.0 - alpha)))
    k = min(max(k, 1), n)
    qhat = float(np.sort(scores)[k - 1])
    return k, qhat

threshold_rows = []
case_table = pd.DataFrame({
    "forecast_case_id": aia_c["forecast_case_id"].to_numpy(),
    "issue_utc": aia_c["issue_utc"].to_numpy() if "issue_utc" in aia_c.columns else pd.NA,
    "label": aia_label,
    "sharp_raw": sharp_p,
    "aia_raw": aia_p,
    "sharp_aia_raw": fusion_p,
})

for branch, p in branches.items():
    p_true = true_class_probability(p, aia_label)
    scores = 1.0 - p_true
    require(((0 <= scores) & (scores <= 1)).all(), f"{branch} nonconformity outside [0,1]")

    case_table[f"{branch}_true_class_probability"] = p_true
    case_table[f"{branch}_nonconformity"] = scores

    for alpha in ALPHAS:
        k, qhat = finite_sample_qhat(scores, alpha)
        min_class_probability = 1.0 - qhat
        threshold_rows.append({
            "branch": branch,
            "alpha": alpha,
            "nominal_coverage": 1.0 - alpha,
            "n_calibration": len(scores),
            "positive_calibration": int(aia_label.sum()),
            "order_statistic_k": k,
            "qhat_nonconformity": qhat,
            "minimum_class_probability_for_set_membership": min_class_probability,
            "nonconformity_definition": "1 - probability_assigned_to_candidate_class",
            "probability_calibration": "raw_identity",
        })

thresholds = pd.DataFrame(threshold_rows)

# We deliberately do NOT score downstream policy-validation coverage here.
# For transparency only, compute calibration-set inclusion/set size diagnostics.
diag_rows = []
for row in threshold_rows:
    branch = row["branch"]
    alpha = row["alpha"]
    qhat = row["qhat_nonconformity"]
    p = branches[branch]

    include_no = p <= qhat                 # score(y=0)=p
    include_yes = (1.0 - p) <= qhat        # score(y=1)=1-p
    set_size = include_no.astype(int) + include_yes.astype(int)
    contains_true = np.where(aia_label == 1, include_yes, include_no)

    diag_rows.append({
        "branch": branch,
        "alpha": alpha,
        "calibration_empirical_true_label_inclusion": float(np.mean(contains_true)),
        "calibration_empty_set_rate": float(np.mean(set_size == 0)),
        "calibration_singleton_rate": float(np.mean(set_size == 1)),
        "calibration_doubleton_rate": float(np.mean(set_size == 2)),
        "note": "descriptive on calibration block only; not downstream coverage evaluation",
    })

diagnostics = pd.DataFrame(diag_rows)

case_path = OUT / "conformal_calibration_cases.csv.gz"
threshold_path = OUT / "thresholds.csv"
diag_path = OUT / "calibration_set_diagnostics.csv"

case_table.to_csv(
    case_path,
    index=False,
    compression={"method": "gzip", "mtime": 0},
)
thresholds.to_csv(threshold_path, index=False)
diagnostics.to_csv(diag_path, index=False)

protocol = {
    "status": "CONFORMAL_THRESHOLDS_FROZEN",
    "scientific_acceptance": False,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "horizon_hours": 72,
    "target": "binary M/X flare occurrence",
    "method": "split_conformal_classification",
    "nonconformity": "1 - probability assigned to candidate class",
    "primary_alpha": PRIMARY_ALPHA,
    "sensitivity_alphas": [0.05, 0.20],
    "alpha_status": "declared now before reading policy_validation outcomes; not part of the earlier probability-calibration config",
    "finite_sample_rule": "k = ceil((n + 1) * (1 - alpha)); qhat = k-th ordered calibration nonconformity score, clipped to n",
    "branches": ["sharp", "aia", "sharp_aia"],
    "probability_calibration": {
        "sharp": "raw_identity",
        "aia": "raw_identity",
        "sharp_aia": "raw_identity",
    },
    "fusion_weights": {"sharp": 0.5, "aia": 0.5},
    "calibration_rows": EXPECTED_ROWS,
    "calibration_positives": EXPECTED_POSITIVES,
    "policy_validation_outcomes_read": False,
    "later_outcomes_read": False,
    "exchangeability_caveat": "Nominal split-conformal coverage is not assumed to remain exact under temporal dependence or distribution shift; later periods are evaluated empirically.",
    "source_sha256": {
        "master_predictions": MASTER_SHA,
        "sharp_predictions": SHARP_SHA,
        "probability_selection_json": sha256(SELECTION),
    },
    "output_sha256": {
        case_path.name: sha256(case_path),
        threshold_path.name: sha256(threshold_path),
        diag_path.name: sha256(diag_path),
    },
}
(OUT / "protocol.json").write_text(json.dumps(protocol, indent=2) + "\n")

print("===== CONFORMAL THRESHOLDS FROZEN =====")
print("rows:", EXPECTED_ROWS)
print("positives:", EXPECTED_POSITIVES)
print("selected probability calibration: raw / raw / raw")
print("\nThresholds:")
print(thresholds.to_string(index=False))
print("\nCalibration-only diagnostics:")
print(diagnostics.to_string(index=False))
print("\nProtocol:")
print(json.dumps(protocol, indent=2))
print("\nSaved:", OUT / "protocol.json")
print("Saved:", threshold_path)
print("Saved:", case_path)
print("Saved:", diag_path)
